# 01 雅可比迭代法与高斯-赛德尔迭代法

本节讨论求解线性方程组

$$
Ax=b
$$

的两种基本平稳迭代法：雅可比迭代法（Jacobi iteration）和高斯-赛德尔迭代法（Gauss-Seidel iteration）。它们的共同思想是把矩阵拆成容易求解的一部分和剩余部分，通过反复更新近似解来降低残差。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    gauss_seidel_iteration,
    gauss_seidel_iteration_matrix,
    is_strictly_diagonally_dominant,
    jacobi_iteration,
    jacobi_iteration_matrix,
    relative_residual,
    spectral_radius,
)


## 矩阵分裂的统一框架

设

$$
A=M-N.
$$

则方程 $Ax=b$ 可以写成

$$
Mx=Nx+b.
$$

若选择一个初值 $x^{(0)}$，就得到定常迭代

$$
Mx^{(k+1)}=Nx^{(k)}+b,
$$

也就是

$$
x^{(k+1)}=B x^{(k)}+c,\qquad B=M^{-1}N,\quad c=M^{-1}b.
$$

若误差 $e^{(k)}=x^{(k)}-x^\ast$，则

$$
e^{(k+1)}=B e^{(k)}.
$$

因此收敛性由迭代矩阵 $B$ 决定。经典结论是：对任意初值收敛当且仅当

$$
\rho(B)<1,
$$

其中 $\rho(B)$ 是谱半径。


## Jacobi 迭代

把

$$
A=D+L+U
$$

分解为对角部分 $D$、严格下三角部分 $L$ 和严格上三角部分 $U$。Jacobi 迭代取

$$
M=D,\qquad N=-(L+U),
$$

得到

$$
Dx^{(k+1)}=b-(L+U)x^{(k)}.
$$

分量形式为

$$
x_i^{(k+1)}=\frac{1}{a_{ii}}\left(b_i-\sum_{j\ne i}a_{ij}x_j^{(k)}\right).
$$

它每个分量都只用上一轮的旧值，因此结构清楚，也便于并行。


In [ ]:
def teaching_jacobi(A, b, x0=None, tolerance=1e-8, max_iterations=200):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.zeros_like(b) if x0 is None else np.asarray(x0, dtype=float).copy()
    D = np.diag(A)
    R = A - np.diag(D)
    residuals = [relative_residual(A, x, b)]
    for k in range(1, max_iterations + 1):
        x = (b - R @ x) / D
        residuals.append(relative_residual(A, x, b))
        if residuals[-1] <= tolerance:
            break
    return x, np.array(residuals)


## Gauss-Seidel 迭代

Gauss-Seidel 迭代把最新得到的分量立即用于后续分量。矩阵形式为

$$
(D+L)x^{(k+1)}=b-Ux^{(k)}.
$$

分量形式为

$$
x_i^{(k+1)}=\frac{1}{a_{ii}}\left(
 b_i-\sum_{j<i}a_{ij}x_j^{(k+1)}-\sum_{j>i}a_{ij}x_j^{(k)}
\right).
$$

它通常比 Jacobi 更快，但更新顺序会影响并行性和某些矩阵上的表现。


In [ ]:
def teaching_gauss_seidel(A, b, x0=None, tolerance=1e-8, max_iterations=200):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.zeros_like(b) if x0 is None else np.asarray(x0, dtype=float).copy()
    residuals = [relative_residual(A, x, b)]
    for k in range(1, max_iterations + 1):
        old = x.copy()
        for i in range(len(b)):
            left = A[i, :i] @ x[:i]
            right = A[i, i + 1 :] @ old[i + 1 :]
            x[i] = (b[i] - left - right) / A[i, i]
        residuals.append(relative_residual(A, x, b))
        if residuals[-1] <= tolerance:
            break
    return x, np.array(residuals)


## 小规模收敛实验

下面使用一个严格对角占优的三对角矩阵。严格对角占优是 Jacobi 和 Gauss-Seidel 收敛的常用充分条件。我们比较两种方法的残差历史，并与解析线性代数求解结果对照。


In [ ]:
A = np.array([
    [4.0, -1.0, 0.0],
    [-1.0, 4.0, -1.0],
    [0.0, -1.0, 4.0],
])
b = np.array([2.0, 4.0, 6.0])
exact = np.linalg.solve(A, b)

x_j, res_j = teaching_jacobi(A, b, tolerance=1e-10)
x_gs, res_gs = teaching_gauss_seidel(A, b, tolerance=1e-10)

print("严格对角占优：", is_strictly_diagonally_dominant(A))
print("Jacobi 误差：", np.linalg.norm(x_j - exact), "迭代次数：", len(res_j) - 1)
print("Gauss-Seidel 误差：", np.linalg.norm(x_gs - exact), "迭代次数：", len(res_gs) - 1)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(res_j, "o-", label="Jacobi")
ax.semilogy(res_gs, "s-", label="Gauss-Seidel")
ax.set_xlabel("迭代次数")
ax.set_ylabel("相对残差")
ax.set_title("Jacobi 与 Gauss-Seidel 残差历史")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 迭代矩阵和谱半径

Jacobi 的迭代矩阵是

$$
B_J=-D^{-1}(L+U),
$$

Gauss-Seidel 的迭代矩阵是

$$
B_{GS}=-(D+L)^{-1}U.
$$

谱半径越小，通常误差衰减越快。这个判断是线性定常迭代的核心，但实际停止时仍应看残差，而不是只看公式阶数。


In [ ]:
B_j = jacobi_iteration_matrix(A)
B_gs = gauss_seidel_iteration_matrix(A)
print("rho(B_J) =", spectral_radius(B_j))
print("rho(B_GS) =", spectral_radius(B_gs))


## 与正式实现对照

Notebook 中的教学实现强调公式对应关系。`src/py_sc/iterative_linear.py` 中的正式实现增加了输入检查、统一结果对象、残差历史和收敛标记。


In [ ]:
formal_j = jacobi_iteration(A, b, tolerance=1e-10, max_iterations=200)
formal_gs = gauss_seidel_iteration(A, b, tolerance=1e-10, max_iterations=200)
print(formal_j)
print(formal_gs)
print("Jacobi 教学/正式差异：", np.linalg.norm(x_j - formal_j.value))
print("GS 教学/正式差异：", np.linalg.norm(x_gs - formal_gs.value))


## 不同初值的影响

对收敛矩阵而言，不同初值最终会收敛到同一个解，但残差曲线前几步可能不同。若迭代矩阵谱半径接近 1，初值差异会更明显。


In [ ]:
initial_guesses = [np.zeros(3), np.array([5.0, -3.0, 2.0]), np.array([-4.0, 4.0, -2.0])]
fig, ax = plt.subplots(figsize=(7, 4))
for idx, x0 in enumerate(initial_guesses, start=1):
    result = jacobi_iteration(A, b, x0=x0, tolerance=1e-10, max_iterations=120)
    ax.semilogy(result.residual_norms, label=f"初值 {idx}")
ax.set_xlabel("迭代次数")
ax.set_ylabel("相对残差")
ax.set_title("不同初值下 Jacobi 残差变化")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 收敛与不收敛对比

若迭代矩阵谱半径大于 1，定常迭代会对某些初值发散。下面构造一个非对角占优矩阵，观察 Jacobi 谱半径和残差变化。


In [ ]:
A_bad = np.array([[1.0, 3.0], [2.0, 1.0]])
b_bad = np.array([1.0, 1.0])
print("rho(B_J bad)=", spectral_radius(jacobi_iteration_matrix(A_bad)))
bad_result = jacobi_iteration(A_bad, b_bad, tolerance=1e-8, max_iterations=20)
print("converged=", bad_result.converged)
print("residuals=", bad_result.residual_norms)


## 本节小结

Jacobi 和 Gauss-Seidel 都来自矩阵分裂。Jacobi 使用上一轮旧值，结构简单；Gauss-Seidel 立即使用新值，常更快。谱半径 $\rho(B)<1$ 是定常迭代收敛的根本条件，严格对角占优是常见充分条件。实际程序中应同时设置残差容差、最大迭代次数和残差历史记录；只看迭代次数或只看相邻差值都不够可靠。
